# Drug Risk Clustering — Portfolio Project
This notebook compares a transparent weighted risk method with K-Means clustering using synthetic drug supply-chain data.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

In [ ]:
DATA_PATH = Path("drug_data.csv")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
feature_columns = ["alternatives", "shortage", "inventory", "demand", "supply"]
X = df[feature_columns].copy()

for column in feature_columns:
    X[column] = pd.to_numeric(X[column], errors="coerce")

X = X.fillna(X.median(numeric_only=True))
X.describe().round(2)

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
signed_weights = {
    "alternatives": -3.0,
    "shortage": 4.0,
    "inventory": -4.0,
    "demand": 2.5,
    "supply": -3.0,
}

signed_vector = np.array([signed_weights[c] for c in feature_columns])
magnitude_vector = np.abs(signed_vector)

df["Risk_Score"] = X_scaled @ signed_vector
X_weighted = X_scaled * magnitude_vector

### Why K-Means uses absolute weight magnitudes
K-Means is distance-based. Multiplying a feature by `-1` only reflects the axis and does not change Euclidean distance. The signs are therefore used for the interpretable risk score, while the magnitudes control how strongly each feature affects clustering.

In [ ]:
p50, p75, p90 = df["Risk_Score"].quantile([0.50, 0.75, 0.90])

def weighted_label(score):
    if score >= p90:
        return "Critical"
    if score >= p75:
        return "High"
    if score >= p50:
        return "Medium"
    return "Low"

df["Drug Cluster"] = df["Risk_Score"].apply(weighted_label)
df["Drug Cluster"].value_counts()

In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=20)
df["Cluster"] = kmeans.fit_predict(X_weighted)

silhouette = silhouette_score(X_weighted, df["Cluster"])
print(f"Silhouette Score: {silhouette:.3f}")

In [ ]:
cluster_order = (
    df.groupby("Cluster")["Risk_Score"]
      .mean()
      .sort_values()
      .index
      .tolist()
)

labels = ["Low", "Medium", "High", "Critical"]
cluster_map = {cluster_order[i]: labels[i] for i in range(4)}
df["Risk_Level"] = df["Cluster"].map(cluster_map)
cluster_map

In [ ]:
df["Same_Label"] = (df["Drug Cluster"] == df["Risk_Level"]).astype(int)
agreement = df["Same_Label"].mean() * 100

confusion = pd.crosstab(
    df["Drug Cluster"],
    df["Risk_Level"],
    rownames=["Weighted Method"],
    colnames=["ML Method"],
).reindex(index=labels, columns=labels, fill_value=0)

print(f"Agreement: {agreement:.2f}%")
confusion

In [ ]:
centers_scaled = pd.DataFrame(
    kmeans.cluster_centers_,
    columns=feature_columns,
)
centers_scaled.index.name = "Cluster"
centers_scaled["Risk_Level"] = centers_scaled.index.map(cluster_map)
centers_scaled

In [ ]:
original_scaled_centers = kmeans.cluster_centers_ / magnitude_vector
original_centers = scaler.inverse_transform(original_scaled_centers)

centers_original = pd.DataFrame(original_centers, columns=feature_columns)
centers_original.index.name = "Cluster"
centers_original["Risk_Level"] = centers_original.index.map(cluster_map)
centers_original.round(2)

In [ ]:
pca = PCA(n_components=2)
pca_values = pca.fit_transform(X_weighted)
df["PCA1"] = pca_values[:, 0]
df["PCA2"] = pca_values[:, 1]

plt.figure(figsize=(10, 7))
for risk_level in labels:
    subset = df[df["Risk_Level"] == risk_level]
    plt.scatter(subset["PCA1"], subset["PCA2"], label=risk_level, alpha=0.65, s=28)
plt.title("Drug Risk Clusters — PCA View")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend(title="ML Risk Level")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
df.to_csv(OUTPUT_DIR / "clustered_drug_results.csv", index=False)
confusion.to_csv(OUTPUT_DIR / "method_confusion_matrix.csv")
centers_scaled.to_csv(OUTPUT_DIR / "weighted_cluster_centers.csv")
centers_original.to_csv(OUTPUT_DIR / "original_scale_cluster_centers.csv")
print("Results exported to the outputs folder.")